# 03 Optimization

This notebook shows one lightweight path for tuning gains with `scipy.optimize` before moving to more advanced robust design workflows.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from src.configuration import load_json_config
from src.control import build_controller
from src.dynamics import VTOLDynamicsModel
from src.optim import tune_gains
from src.simulations.core import simulate_experiment
from src.simulations.trajectories import hover_trajectory


In [ ]:
config = load_json_config(REPO_ROOT / "config" / "default_params.json")
model = VTOLDynamicsModel.from_config(config)
dt = config["simulations"]["dt"]
duration = 8.0
initial_state = np.zeros(12)
initial_state[2] = -2.0
initial_state[0] = 1.0

def dynamics_runner(gains):
    tuned = load_json_config(REPO_ROOT / "config" / "default_params.json")
    tuned["controllers"]["pid"]["velocity_gains"]["kp"] = [gains[0], gains[0], gains[1]]
    controller = build_controller("pid", model, tuned)
    return simulate_experiment(model, controller, None, hover_trajectory(position=(0.0, 0.0, -2.0)), initial_state, duration, dt)

def cost_fn(dynamics_fn, gains):
    results = dynamics_fn(gains)
    tracking_cost = np.mean(np.linalg.norm(results.errors[:, 0:3], axis=1))
    control_cost = 0.01 * np.mean(np.linalg.norm(results.control, axis=1))
    return tracking_cost + control_cost

result = tune_gains(dynamics_runner, cost_fn, initial_guess=[0.8, 1.2], bounds=[(0.1, 2.0), (0.1, 3.0)])
result


**Research note:** Even this lightweight optimization loop is enough to generate defensible baseline comparisons, especially when paired with transparent cost definitions that separate tracking, effort, and fault sensitivity.
